# library

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import pandas as pd
import os,sys
import seaborn as sns
from scipy import sparse
import matplotlib.cm as cm
import biolucid 
from tidyi import tidyi_main_mouse
from scipy.stats import pearsonr
from pathlib import Path

# loading data

In [2]:
# loading data which annotated by ASTRID
ASTRID_0204_DIRS = {
    'tms_bm': '/scratch/chenxi.sun/thesis_project/tms_qcdata/AllRef_BM_0204',
    'mukin_bm': '/scratch/chenxi.sun/thesis_project/mukin_qcdata/AllRef_BM_0204',
    'mukin_sk': '/scratch/chenxi.sun/thesis_project/mukin_qcdata/AllRef_skin_0204',
}

def load_merge_adata(data_dir, file_pattern='*outASTRID.h5ad'):
    """Load one ASTRID h5ad per sample subdirectory and merge them."""
    data_dir = Path(data_dir)
    adatas = []

    for sample_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        files = sorted(sample_dir.glob(file_pattern))
        if files:
            adatas.append(sc.read_h5ad(files[0]))

    if not adatas:
        raise FileNotFoundError(f'No {file_pattern} files found under {data_dir}')

    return adatas[0].concatenate(adatas[1:], join='inner')


In [3]:
tms_bm_astrid = load_merge_adata(ASTRID_0204_DIRS['tms_bm'])
mukin_bm_astrid = load_merge_adata(ASTRID_0204_DIRS['mukin_bm'])
mukin_sk_astrid = load_merge_adata(ASTRID_0204_DIRS['mukin_sk'])

In [4]:
tms_bm_astrid.obs

,age,cell_ontology_class,cell_ontology_id,free_annotation,method,mouse.id,n_genes,obs_names,sex,subtissue,...,clustering_level_6,final_clustering_level,SingleR_Pruned_CellType,SingleR_All_deltanext,SingleR_CellType,clustering_level_7,clustering_level_8,clustering_level_9,clustering_level_10,batch
10X_P5_10_AAACCTGAGACAGGCT-0,NaN,NaN,NaN,NaN,NaN,1-M-62,702,NaN,NaN,NaN,...,1_0_3_0_0,1_0_3_0_0,erythrocyte,0.358783,erythrocyte,NaN,NaN,NaN,NaN,0
10X_P5_10_AAACCTGGTGCAGTAG-0,1m,granulocytopoietic cell,NA,nan,droplet,1-M-62,1677,AAACCTGGTGCAGTAG-1-53-1-0,male,BM,...,1_1_3_1,1_1_3_1,erythrocyte,0.204261,erythrocyte,NaN,NaN,NaN,NaN,0
10X_P5_10_AAACCTGGTGTGACCC-0,1m,naive T cell,NA,nan,droplet,1-M-62,5726,AAACCTGGTGTGACCC-1-53-1-0,male,BM,...,0_4_0,0_4_0,neutrophil progenitor,0.674570,neutrophil progenitor,NaN,NaN,NaN,NaN,0
10X_P5_10_AAACCTGTCGCTTGTC-0,NaN,NaN,NaN,NaN,NaN,1-M-62,488,NaN,NaN,NaN,...,2_0_0_1_1,2_0_0_1_1,neutrophil,0.325601,neutrophil,NaN,NaN,NaN,NaN,0
10X_P5_10_AAACGGGAGACCGGAT-0,1m,monocyte,NA,nan,droplet,1-M-62,2118,AAACGGGAGACCGGAT-1-53-1-0,male,BM,...,0_3_1_1,0_3_1_1,monocyte,0.081036,monocyte,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10X_P3_5_TTTGTCAGTGATGTCT-14,NaN,NaN,NaN,NaN,NaN,30-M-5,642,NaN,NaN,NaN,...,1_1_0_3_1,1_1_0_3_1,neutrophil,0.674693,neutrophil,1_1_0_3_1,1_1_0_3_1,NaN,NaN,14
10X_P3_5_TTTGTCATCCAAGCCG-14,NaN,NaN,NaN,NaN,NaN,30-M-5,952,NaN,NaN,NaN,...,0_1_0_1_1,0_1_0_1_1,B naive,0.588806,B naive,0_1_0_1_1,0_1_0_1_1,NaN,NaN,14
10X_P3_5_TTTGTCATCTCTGAGA-14,30m,hematopoietic precursor cell,NA,nan,droplet,30-M-5,1536,TTTGTCATCTCTGAGA-1-38-1-0,male,BM,...,0_2_0_1_0,0_2_0_1_0,T CD8,0.069672,T CD8,0_2_0_1_0,0_2_0_1_0,NaN,NaN,14
10X_P3_5_TTTGTCATCTGCCCTA-14,NaN,NaN,NaN,NaN,NaN,30-M-5,479,NaN,NaN,NaN,...,1_2_1_0_0,1_2_1_0_0,neutrophil,0.763286,neutrophil,1_2_1_0_0,1_2_1_0_0,NaN,NaN,14


# reannotate those contaminated RBC cell

In [5]:
# contaminated RBC reannotation
RBC_MARKER_GENES = ['Hbb-bs', 'Hbb-bt', 'Hba-a1', 'Hba-a2', 'Hbb-bh1', 'Hbb-bh2']

ASTRID_RESULT_FILES = {
    'mukin_bm': '/scratch/chenxi.sun/thesis_project/mukin_qcdata/AllRef_BM_0204/BM_combined_ASTRID_results.csv',
}

TMS_ERYTHROCYTE_RULES = {
    '1-M-62': ['1_0', '1_1', '1_2', '1_3'],
    '1-M-63': ['3'],
    '3-F-56': ['0_1', '0_6'],
    '3-F-57': ['2'],
    '18-M-52': ['2'],
    '18-M-53': ['2_2', '2_4'],
    '21-F-54': ['0_9'],
    '21-F-55': ['1_2', '1_4'],
    '24-M-58': ['2', '3'],
    '24-M-59': ['1_1'],
    '24-M-60': ['1', '3'],
    '24-M-61': ['1', '3'],
    '30-M-3': ['2', '3'],
    '30-M-2': ['1', '4'],
    '30-M-5': ['2', '3'],
}

def _set_obs_values(adata, column, mask, values):
    if pd.api.types.is_categorical_dtype(adata.obs[column]):
        new_cats = pd.Index(pd.Series(values).unique()).union(adata.obs[column].cat.categories)
        adata.obs[column] = adata.obs[column].cat.set_categories(new_cats)
    adata.obs.loc[mask, column] = values

def _deepest_cluster_level(row):
    cluster_cols = sorted(
        [col for col in row.index if col.startswith('clustering_level_')],
        key=lambda col: int(col.rsplit('_', 1)[1]),
        reverse=True,
    )
    for col in cluster_cols:
        if pd.notna(row[col]):
            return str(row[col])
    return None

def reannotate_rbc_contaminated_clusters(adata, astrid_result_csv, sample_col, source_col='SingleR_CellType', target_col='celltype_hb'):
    adata.obs[target_col] = adata.obs[source_col].copy()
    result_df = pd.read_csv(astrid_result_csv)
    rbc_mask = result_df['OddsRatioMarkerGenes'].apply(
        lambda genes: any(gene in str(genes) for gene in RBC_MARKER_GENES)
    )
    rbc_clusters = result_df.loc[rbc_mask].copy()
    rbc_clusters['clustering_level'] = rbc_clusters.apply(_deepest_cluster_level, axis=1)
    rbc_clusters = rbc_clusters.dropna(subset=['sample_id', 'clustering_level'])

    annotated_cells = 0
    for _, row in rbc_clusters.iterrows():
        mask = (
            (adata.obs[sample_col].astype(str) == str(row['sample_id']))
            & (adata.obs['final_clustering_level'].astype(str) == str(row['clustering_level']))
        )
        if mask.any():
            old_values = adata.obs.loc[mask, target_col].astype(str)
            new_values = np.where(
                old_values.str.startswith('RBC_contam_'),
                old_values,
                'RBC_contam_' + old_values,
            )
            _set_obs_values(adata, target_col, mask, new_values)
            annotated_cells += int(mask.sum())

    erythrocyte_mask = adata.obs[target_col].astype(str) == 'RBC_contam_erythrocyte'
    if erythrocyte_mask.any():
        _set_obs_values(adata, target_col, erythrocyte_mask, ['erythrocyte'] * int(erythrocyte_mask.sum()))

    return rbc_clusters[['sample_id', 'clustering_level', 'SingleR_CellType', 'CellCount']], annotated_cells

def reannotate_tms_erythrocyte_clusters(adata):
    adata.obs['celltype_hb'] = adata.obs['SingleR_CellType'].copy()
    annotated_cells = 0

    for mouse_id, prefixes in TMS_ERYTHROCYTE_RULES.items():
        mouse_mask = adata.obs['mouse.id'].astype(str) == mouse_id
        cluster_values = adata.obs.loc[mouse_mask, 'final_clustering_level'].astype(str)
        cluster_mask = cluster_values.apply(lambda cluster: any(cluster.startswith(prefix) for prefix in prefixes))
        mask = mouse_mask.copy()
        mask.loc[mouse_mask] = cluster_mask.values

        if mask.any():
            _set_obs_values(adata, 'celltype_hb', mask, ['erythrocyte'] * int(mask.sum()))
            annotated_cells += int(mask.sum())

    return annotated_cells

tms_erythrocyte_cells = reannotate_tms_erythrocyte_clusters(tms_bm_astrid)
mukin_bm_rbc_clusters, mukin_bm_rbc_cells = reannotate_rbc_contaminated_clusters(
    mukin_bm_astrid,
    ASTRID_RESULT_FILES['mukin_bm'],
    sample_col='mouseID',
)

pd.DataFrame({
    'dataset': ['tms_bm', 'mukin_bm'],
    'reannotated_cells': [tms_erythrocyte_cells, mukin_bm_rbc_cells],
})

/tmp/ipykernel_736417/4028391568.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata.obs[column]):
/tmp/ipykernel_736417/4028391568.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata.obs[column]):
/tmp/ipykernel_736417/4028391568.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata.obs[column]):
/tmp/ipykernel_736417/4028391568.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata.obs[column]):
/tmp/ipykernel_7

,dataset,reannotated_cells
0,tms_bm,15511
1,mukin_bm,1267


# TIDYI pipeline

## check the sample-celltype cell counts distribution

In [6]:
# check the sample-celltype cell count distribution for each data
ADATA_COUNT_CONFIG = {
    'tms_bm': (tms_bm_astrid, 'sample', 'celltype_hb'),
    'mukin_bm': (mukin_bm_astrid, 'Sample', 'celltype_hb'),
    'mukin_sk': (mukin_sk_astrid, 'Sample', 'SingleR_CellType'),
}

def count_cells_by_sample_celltype(adata, sample_col, celltype_col):
    return (
        adata.obs
        .groupby([sample_col, celltype_col], observed=True)
        .size()
        .reset_index(name='cell_count')
        .sort_values([sample_col, 'cell_count'], ascending=[True, False])
        .reset_index(drop=True)
    )

sample_celltype_counts = []
sample_celltype_count_tables = {}

for dataset, (adata, sample_col, celltype_col) in ADATA_COUNT_CONFIG.items():
    counts = count_cells_by_sample_celltype(adata, sample_col, celltype_col)
    counts.insert(0, 'dataset', dataset)
    counts = counts.rename(columns={sample_col: 'sample', celltype_col: 'celltype'})

    sample_celltype_counts.append(counts)
    sample_celltype_count_tables[dataset] = counts.pivot(
        index='sample', columns='celltype', values='cell_count'
    ).fillna(0).astype(int)

sample_celltype_counts = pd.concat(sample_celltype_counts, ignore_index=True)
sample_celltype_counts

,dataset,sample,celltype,cell_count
0,tms_bm,10X_P1_5,neutrophil maturing,4429
1,tms_bm,10X_P1_5,erythrocyte,3799
2,tms_bm,10X_P1_5,neutrophil,2235
3,tms_bm,10X_P1_5,neutrophil immature,2028
4,tms_bm,10X_P1_5,monocyte,496
...,...,...,...,...
755,mukin_sk,A210,hair follicle bulge,140
756,mukin_sk,A210,hair follicle,48
757,mukin_sk,A210,T epithelium,41
758,mukin_sk,A210,langerhans,38


In [7]:
# before run TIDYI, filter out cell types with fewer than 100 cells in each sample
MIN_CELLS_PER_SAMPLE_CELLTYPE = 100

def filter_min_cells_per_sample_celltype(adata, sample_col, celltype_col, min_cells=100):
    group_counts = adata.obs.groupby([sample_col, celltype_col], observed=True)[celltype_col].transform('size')
    keep_mask = group_counts >= min_cells
    return adata[keep_mask].copy(), pd.DataFrame({
        'cells_before': [adata.n_obs],
        'cells_after': [int(keep_mask.sum())],
        'cells_removed': [int((~keep_mask).sum())],
        'groups_before': [adata.obs[[sample_col, celltype_col]].drop_duplicates().shape[0]],
        'groups_after': [adata.obs.loc[keep_mask, [sample_col, celltype_col]].drop_duplicates().shape[0]],
    })

filtered_adatas = {}
filter_summaries = []

for dataset, (adata, sample_col, celltype_col) in ADATA_COUNT_CONFIG.items():
    filtered_adata, summary = filter_min_cells_per_sample_celltype(
        adata,
        sample_col=sample_col,
        celltype_col=celltype_col,
        min_cells=MIN_CELLS_PER_SAMPLE_CELLTYPE,
    )
    filtered_adatas[dataset] = filtered_adata
    summary.insert(0, 'dataset', dataset)
    filter_summaries.append(summary)

tms_bm_filtered = filtered_adatas['tms_bm']
mukin_bm_filtered = filtered_adatas['mukin_bm']
mukin_sk_filtered = filtered_adatas['mukin_sk']

filter_summary = pd.concat(filter_summaries, ignore_index=True)
filter_summary

,dataset,cells_before,cells_after,cells_removed,groups_before,groups_after
0,tms_bm,75456,68205,7251,312,156
1,mukin_bm,14193,8014,6179,195,47
2,mukin_sk,48267,43814,4453,253,98


## Run TIDYI based on animal-celltype

In [ ]:
# run TIDYI based on animal-celltype
import mygene

TIDYI_RUN_CONFIG = {
    'tms_bm': {
        'adata': tms_bm_filtered,
        'age_col': 'age',
        'animal_col': 'mouse.id',
        'celltype_col': 'celltype_hb',
    },
    'mukin_bm': {
        'adata': mukin_bm_filtered,
        'age_col': 'age',
        'animal_col': 'mouseID',
        'celltype_col': 'celltype_hb',
    },
    'mukin_sk': {
        'adata': mukin_sk_filtered,
        'age_col': 'age',
        'animal_col': 'mouseID',
        'celltype_col': 'SingleR_CellType',
    },
}

def map_mouse_symbols_to_ensembl(adata):
    mg = mygene.MyGeneInfo()
    current_genes = adata.var_names.tolist()
    gene_info = mg.querymany(
        current_genes,
        scopes='symbol',
        species='mouse',
        fields='ensembl.gene',
        as_dataframe=True,
        verbose=False,
    )

    gene_info['symbol'] = gene_info.index
    mapping_df = gene_info.dropna(subset=['ensembl.gene']).drop_duplicates(subset=['symbol'])

    gene_mapping = {}
    for symbol, ensembl_gene in zip(mapping_df['symbol'], mapping_df['ensembl.gene']):
        if isinstance(ensembl_gene, list):
            ensembl_gene = ensembl_gene[0]
        if isinstance(ensembl_gene, dict):
            ensembl_gene = ensembl_gene.get('gene')
        if isinstance(ensembl_gene, str):
            gene_mapping[symbol] = ensembl_gene

    adata.var_names = [gene_mapping.get(gene, gene) for gene in current_genes]
    if adata.var_names.duplicated().any():
        adata.var_names_make_unique()

def run_tidyi_by_animal_celltype(adata, age_col, animal_col, celltype_col, dataset):
    adata = adata.copy()
    map_mouse_symbols_to_ensembl(adata)

    adata.obs['anim_ct'] = (
        adata.obs[animal_col].astype(str) + '__' + adata.obs[celltype_col].astype(str)
    )

    adata.layers['counts'] = adata.layers['raw'].copy()

    tm = tidyi_main_mouse.tidy_rates_mouse(gene_list=adata.var_names.tolist())
    tm.train_classifier()

    results = []
    for group in sorted(adata.obs['anim_ct'].unique()):
        group_mask = adata.obs['anim_ct'] == group
        animal, celltype = group.split('__', 1)
        group_obs = adata.obs.loc[group_mask]
        age = group_obs[age_col].astype(str).iloc[0]

        prolif_rate, death_rate, _, _ = tm.apply_classifier(adata[group_mask].copy())
        results.append({
            'dataset': dataset,
            'sample': group,
            'animal': animal,
            'age': age,
            'celltype': celltype,
            'cell_count': int(group_mask.sum()),
            'prolif_rate': float(prolif_rate),
            'death_rate': float(death_rate),
            'growth_rate': float(prolif_rate - death_rate),
            'turn_over': float((np.sqrt(2) / 2) * (prolif_rate + death_rate)),
        })

    return pd.DataFrame(results)

def fill_tidyi_metadata(result_df):
    result_df = result_df.copy()
    animal = result_df['animal'].astype(str)
    missing_age = result_df['age'].isna() | result_df['age'].astype(str).str.lower().isin(['', 'nan', 'none'])

    tms_mask = result_df['dataset'].str.startswith('tms')
    tms_meta = animal.str.extract(r'^(?P<age>\d+)-(?P<sex>[A-Za-z]+)-(?P<animal_number>.+)$')
    result_df.loc[tms_mask & missing_age, 'age'] = tms_meta.loc[tms_mask & missing_age, 'age'] + 'm'
    result_df.loc[tms_mask, 'sex'] = tms_meta.loc[tms_mask, 'sex'].str.upper()
    result_df.loc[tms_mask, 'microbiota'] = 'NA'

    mukin_mask = result_df['dataset'].str.startswith('mukin')
    mukin_meta = animal.str.extract(r'^(?P<microbiota>SPF|GF)_(?P<age>\d+m)_(?P<animal_number>.+)$')
    result_df.loc[mukin_mask & missing_age, 'age'] = mukin_meta.loc[mukin_mask & missing_age, 'age']
    result_df.loc[mukin_mask, 'sex'] = 'F'
    result_df.loc[mukin_mask, 'microbiota'] = mukin_meta.loc[mukin_mask, 'microbiota']

    result_df['tissue'] = result_df['dataset'].str.split('_').str[-1].str.upper()
    return result_df

tidyi_result_df = pd.concat(
    [
        run_tidyi_by_animal_celltype(dataset=dataset, **config)
        for dataset, config in TIDYI_RUN_CONFIG.items()
    ],
    ignore_index=True,
)
tidyi_result_df = fill_tidyi_metadata(tidyi_result_df)

tidyi_result_df.to_csv('/scratch/chenxi.sun/RA project/rates reproduce/tidyi_result_df.csv', index=False)
tidyi_result_df

,dataset,sample,animal,age,celltype,cell_count,prolif_rate,death_rate,growth_rate,turn_over,sex,microbiota,tissue
0,tms_bm,1-M-62__B immature,1-M-62,1m,B immature,416,0.003070,0.024129,-0.021060,0.019233,M,NA,BM
1,tms_bm,1-M-62__B naive,1-M-62,1m,B naive,252,0.002495,0.023173,-0.020677,0.018150,M,NA,BM
2,tms_bm,1-M-62__B pre,1-M-62,1m,B pre,175,0.115621,0.011321,0.104300,0.089762,M,NA,BM
3,tms_bm,1-M-62__erythrocyte,1-M-62,1m,erythrocyte,1063,0.038960,0.006466,0.032494,0.032121,M,NA,BM
4,tms_bm,1-M-62__monocyte,1-M-62,1m,monocyte,176,0.011898,0.012829,-0.000930,0.017484,M,NA,BM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,mukin_sk,SPF_2m_5__keratinocyte basal label retain,SPF_2m_5,2m,keratinocyte basal label retain,831,0.006956,0.014452,-0.007495,0.015138,F,SPF,SK
282,mukin_sk,SPF_2m_5__keratinocyte granulosum,SPF_2m_5,2m,keratinocyte granulosum,508,0.010993,0.020688,-0.009695,0.022403,F,SPF,SK
283,mukin_sk,SPF_2m_5__keratinocyte proliferating,SPF_2m_5,2m,keratinocyte proliferating,270,0.126093,0.008271,0.117822,0.095009,F,SPF,SK
284,mukin_sk,SPF_2m_5__keratinocyte spinosum suprabasal,SPF_2m_5,2m,keratinocyte spinosum suprabasal,339,0.007834,0.013722,-0.005887,0.015243,F,SPF,SK


# replicate reproducible scatter plot

In [4]:
# table of comparable mouse groups within each dataset
pair_group_cols = ['dataset', 'age', 'sex', 'microbiota', 'tissue']
if not set(pair_group_cols).issubset(tidyi_result_df.columns):
    tidyi_result_df = fill_tidyi_metadata(tidyi_result_df)

mouse_condition_table = (
    tidyi_result_df
    .groupby(pair_group_cols + ['animal'], dropna=False)
    .agg(
        celltypes_after_filter=('celltype', lambda values: sorted(values.dropna().unique())),
        n_celltypes_after_filter=('celltype', 'nunique'),
    )
    .reset_index()
)

paired_mouse_groups = (
    mouse_condition_table
    .groupby(pair_group_cols, dropna=False)
    .agg(
        n_animals=('animal', 'nunique'),
        animals=('animal', lambda values: ', '.join(sorted(values.astype(str).unique()))),
        n_celltypes_per_animal=('n_celltypes_after_filter', lambda values: ', '.join(map(str, values.tolist()))),
    )
    .reset_index()
)
paired_mouse_groups = paired_mouse_groups[paired_mouse_groups['n_animals'] >= 2]
paired_mouse_groups

,dataset,age,sex,microbiota,tissue,n_animals,animals,n_celltypes_per_animal
0,mukin_bm,19m,F,GF,BM,2,"GF_19m_2, GF_19m_3","3, 5"
1,mukin_bm,19m,F,SPF,BM,3,"SPF_19m_1, SPF_19m_2, SPF_19m_3","6, 6, 6"
2,mukin_bm,2m,F,GF,BM,2,"GF_2m_2, GF_2m_3","4, 5"
3,mukin_bm,2m,F,SPF,BM,3,"SPF_2m_3, SPF_2m_4, SPF_2m_5","2, 6, 4"
4,mukin_sk,19m,F,GF,SK,3,"GF_19m_1, GF_19m_2, GF_19m_3","5, 8, 7"
5,mukin_sk,19m,F,SPF,SK,3,"SPF_19m_1, SPF_19m_2, SPF_19m_3","6, 7, 8"
6,mukin_sk,2m,F,GF,SK,3,"GF_2m_1, GF_2m_2, GF_2m_3","4, 7, 8"
7,mukin_sk,2m,F,SPF,SK,3,"SPF_2m_3, SPF_2m_4, SPF_2m_5","7, 8, 8"
8,tms_bm,18m,M,NaN,BM,2,"18-M-52, 18-M-53","10, 7"
9,tms_bm,1m,M,NaN,BM,2,"1-M-62, 1-M-63","9, 10"


In [5]:
# comparison filtration
# Keep only the two mice with the largest total cell count in each condition,
# then keep comparisons with more than 3 shared celltypes.
tidyi_result_df = pd.read_csv("/scratch/chenxi.sun/RA project/rates reproduce/tidyi_result_df.csv")
animal_cell_counts = (
    tidyi_result_df
    .groupby(pair_group_cols + ['animal'], dropna=False)
    .agg(total_cell_count=('cell_count', 'sum'))
    .reset_index()
)

mouse_condition_ranked = (
    mouse_condition_table
    .merge(animal_cell_counts, on=pair_group_cols + ['animal'], how='left')
    .sort_values(pair_group_cols + ['total_cell_count', 'animal'], ascending=[True, True, True, True, True, False, True])
)
mouse_condition_ranked['condition_cell_count_rank'] = (
    mouse_condition_ranked
    .groupby(pair_group_cols, dropna=False)['total_cell_count']
    .rank(method='first', ascending=False)
    .astype(int)
)

top_two_mouse_condition_table = mouse_condition_ranked[
    mouse_condition_ranked['condition_cell_count_rank'] <= 2
].copy()

paired_mouse_groups = (
    top_two_mouse_condition_table
    .groupby(pair_group_cols, dropna=False)
    .agg(
        n_animals=('animal', 'nunique'),
        animals=('animal', lambda values: ', '.join(values.astype(str).tolist())),
        total_cell_counts=('total_cell_count', lambda values: ', '.join(map(str, values.astype(int).tolist()))),
        n_celltypes_per_animal=('n_celltypes_after_filter', lambda values: ', '.join(map(str, values.tolist()))),
        shared_celltypes=('celltypes_after_filter', lambda values: sorted(set.intersection(*map(set, values))) if len(values) >= 2 else []),
    )
    .reset_index()
)
paired_mouse_groups['n_shared_celltypes'] = paired_mouse_groups['shared_celltypes'].str.len()
paired_mouse_groups['shared_celltypes'] = paired_mouse_groups['shared_celltypes'].apply(lambda values: ', '.join(values))
paired_mouse_groups = paired_mouse_groups[
    (paired_mouse_groups['n_animals'] == 2)
    & (paired_mouse_groups['n_shared_celltypes'] > 3)
].reset_index(drop=True)
paired_mouse_groups


,dataset,age,sex,microbiota,tissue,n_animals,animals,total_cell_counts,n_celltypes_per_animal,shared_celltypes,n_shared_celltypes
0,mukin_bm,19m,F,SPF,BM,2,"SPF_19m_2, SPF_19m_1","1288, 1149","6, 6","T CD8, dendritic, monocyte, monocyte pro, neut...",5
1,mukin_bm,2m,F,GF,BM,2,"GF_2m_3, GF_2m_2","957, 667","5, 4","B immature, B pre, monocyte, neutrophil maturing",4
2,mukin_bm,2m,F,SPF,BM,2,"SPF_2m_4, SPF_2m_5","847, 734","6, 4","B immature, monocyte, neutrophil immature, neu...",4
3,mukin_sk,19m,F,GF,SK,2,"GF_19m_1, GF_19m_3","6368, 3835","5, 7","hair follicle bulge, keratinocyte basal label ...",4
4,mukin_sk,19m,F,SPF,SK,2,"SPF_19m_2, SPF_19m_3","4317, 3329","7, 8","hair follicle, keratinocyte basal early respon...",7
5,mukin_sk,2m,F,GF,SK,2,"GF_2m_3, GF_2m_1","3369, 3272","8, 4","keratinocyte basal label retain, keratinocyte ...",4
6,mukin_sk,2m,F,SPF,SK,2,"SPF_2m_4, SPF_2m_5","3945, 3698","8, 8","hair follicle, hair follicle bulge, keratinocy...",8
7,tms_bm,18m,M,NaN,BM,2,"18-M-52, 18-M-53","3580, 1283","10, 7","erythrocyte, macrophage, neutrophil, neutrophi...",6
8,tms_bm,1m,M,NaN,BM,2,"1-M-62, 1-M-63","3297, 3041","9, 10","B immature, B naive, B pre, erythrocyte, monoc...",9
9,tms_bm,21m,F,NaN,BM,2,"21-F-54, 21-F-55","4450, 4032","14, 12","B age, B naive, endothelial, erythrocyte, fibr...",12


In [7]:
# scatter plots for replicate consistency within comparable mouse groups
from itertools import combinations
from pathlib import Path
from matplotlib.lines import Line2D
import re

SCATTERPLOT_DIR = Path('/scratch/chenxi.sun/RA project/rates reproduce/rep_align_scatterplot copy')
SCATTERPLOT_DIR.mkdir(parents=True, exist_ok=True)

MIN_SHARED_CELLTYPES = 3
RATE_SPECS = [
    ('prolif_rate', 'Proliferation rate'),
    ('death_rate', 'Death rate'),
]

CELLTYPE_MARKERS = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', '*', 'h', 'H']
CELLTYPE_COLORS = list(plt.cm.tab20.colors) + list(plt.cm.Set3.colors) + list(plt.cm.Dark2.colors)
CELLTYPE_STYLE = {
    celltype: {
        'color': CELLTYPE_COLORS[i % len(CELLTYPE_COLORS)],
        'marker': CELLTYPE_MARKERS[(i // len(CELLTYPE_COLORS)) % len(CELLTYPE_MARKERS)],
    }
    for i, celltype in enumerate(sorted(tidyi_result_df['celltype'].dropna().unique()))
}

def _safe_filename(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def _rates_for_pair(result_df, mouse_a, mouse_b):
    rates_a = result_df[result_df['animal'].astype(str) == str(mouse_a)][[
        'celltype', 'prolif_rate', 'death_rate', 'cell_count'
    ]].rename(columns={
        'prolif_rate': 'prolif_rate_a',
        'death_rate': 'death_rate_a',
        'cell_count': 'cell_count_a',
    })
    rates_b = result_df[result_df['animal'].astype(str) == str(mouse_b)][[
        'celltype', 'prolif_rate', 'death_rate', 'cell_count'
    ]].rename(columns={
        'prolif_rate': 'prolif_rate_b',
        'death_rate': 'death_rate_b',
        'cell_count': 'cell_count_b',
    })
    return rates_a.merge(rates_b, on='celltype', how='inner')

def _pearson_r(pair_df, rate_col):
    return pair_df[f'{rate_col}_a'].corr(pair_df[f'{rate_col}_b'], method='pearson')

def _format_pearson(value):
    return 'NA' if pd.isna(value) else f'{value:.2f}'

def _plot_rate_pair(pair_df, dataset, condition_label, mouse_a, mouse_b, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)

    for ax, (rate_col, title) in zip(axes, RATE_SPECS):
        x_col = f'{rate_col}_a'
        y_col = f'{rate_col}_b'
        for _, row in pair_df.iterrows():
            style = CELLTYPE_STYLE[row['celltype']]
            ax.scatter(
                row[x_col],
                row[y_col],
                s=54,
                alpha=0.85,
                color=style['color'],
                marker=style['marker'],
                edgecolor='black',
                linewidth=0.35,
            )

        lim_max = max(float(pair_df[x_col].max()), float(pair_df[y_col].max())) * 1.08 if len(pair_df) else 1
        ax.plot([0, lim_max], [0, lim_max], linestyle='--', color='black', linewidth=1, alpha=0.5)
        ax.set_xlim(0, lim_max)
        ax.set_ylim(0, lim_max)
        ax.set_xlabel(str(mouse_a))
        ax.set_ylabel(str(mouse_b))
        ax.set_title(title)

        pearson_label = _format_pearson(_pearson_r(pair_df, rate_col))
        ax.text(
            0.05,
            0.95,
            f'Pearson r = {pearson_label}',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=9,
            bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.8},
        )

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker=CELLTYPE_STYLE[celltype]['marker'],
            color='none',
            markerfacecolor=CELLTYPE_STYLE[celltype]['color'],
            markeredgecolor='black',
            markeredgewidth=0.35,
            markersize=7,
            label=celltype,
        )
        for celltype in sorted(pair_df['celltype'].unique())
    ]
    fig.legend(
        handles=legend_handles,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        fontsize=7,
    )

    fig.suptitle(f'{dataset}: {condition_label}\n{mouse_a} vs {mouse_b} ({len(pair_df)} shared celltypes)')
    filename = '_'.join(map(_safe_filename, [dataset, condition_label, mouse_a, 'vs', mouse_b])) + '.png'
    output_path = output_dir / filename
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return output_path

def _pearson_rows(pair_df, dataset, condition_label, mouse_a, mouse_b):
    rows = []
    for rate_col, _ in RATE_SPECS:
        rows.append({
            'dataset': dataset,
            'condition': condition_label,
            'mouse_a': mouse_a,
            'mouse_b': mouse_b,
            'rate': rate_col,
            'n_shared_celltypes': pair_df.shape[0],
            'pearson_r': _pearson_r(pair_df, rate_col),
        })
    return rows

pairwise_rate_tables = []
pearson_rows = []
saved_figure_paths = []
skipped_pairs = []

for _, group in paired_mouse_groups.iterrows():
    dataset = group['dataset']
    dataset_rates = tidyi_result_df[tidyi_result_df['dataset'] == dataset]
    animals = [animal.strip() for animal in group['animals'].split(',')]
    condition_label = ', '.join(
        f'{col}={group[col]}' for col in ['age', 'sex', 'microbiota', 'tissue']
    )

    for mouse_a, mouse_b in combinations(animals, 2):
        pair_df = _rates_for_pair(dataset_rates, mouse_a, mouse_b)
        n_shared_celltypes = pair_df.shape[0]
        if n_shared_celltypes <= MIN_SHARED_CELLTYPES:
            skipped_pairs.append({
                'dataset': dataset,
                'condition': condition_label,
                'mouse_a': mouse_a,
                'mouse_b': mouse_b,
                'n_shared_celltypes': n_shared_celltypes,
                'reason': f'n_shared_celltypes <= {MIN_SHARED_CELLTYPES}',
            })
            continue

        pair_df.insert(0, 'mouse_b', mouse_b)
        pair_df.insert(0, 'mouse_a', mouse_a)
        pair_df.insert(0, 'condition', condition_label)
        pair_df.insert(0, 'dataset', dataset)
        pairwise_rate_tables.append(pair_df)
        pearson_rows.extend(_pearson_rows(pair_df, dataset, condition_label, mouse_a, mouse_b))

        saved_figure_paths.append(_plot_rate_pair(pair_df, dataset, condition_label, mouse_a, mouse_b, SCATTERPLOT_DIR))

pairwise_rate_comparisons = pd.concat(pairwise_rate_tables, ignore_index=True) if pairwise_rate_tables else pd.DataFrame()
pairwise_rate_pearson_correlations = pd.DataFrame(pearson_rows)
skipped_pairwise_rate_comparisons = pd.DataFrame(skipped_pairs)

pairwise_rate_pearson_correlations.to_csv(SCATTERPLOT_DIR / 'pairwise_rate_pearson_correlations.csv', index=False)
pairwise_rate_comparisons.to_csv(SCATTERPLOT_DIR / 'pairwise_rate_comparisons.csv', index=False)
skipped_pairwise_rate_comparisons.to_csv(SCATTERPLOT_DIR / 'skipped_pairwise_rate_comparisons.csv', index=False)
pairwise_rate_pearson_correlations


,dataset,condition,mouse_a,mouse_b,rate,n_shared_celltypes,pearson_r
0,mukin_bm,"age=19m, sex=F, microbiota=SPF, tissue=BM",SPF_19m_2,SPF_19m_1,prolif_rate,5,0.990802
1,mukin_bm,"age=19m, sex=F, microbiota=SPF, tissue=BM",SPF_19m_2,SPF_19m_1,death_rate,5,0.971634
2,mukin_bm,"age=2m, sex=F, microbiota=GF, tissue=BM",GF_2m_3,GF_2m_2,prolif_rate,4,0.999640
3,mukin_bm,"age=2m, sex=F, microbiota=GF, tissue=BM",GF_2m_3,GF_2m_2,death_rate,4,0.889293
4,mukin_bm,"age=2m, sex=F, microbiota=SPF, tissue=BM",SPF_2m_4,SPF_2m_5,prolif_rate,4,0.998345
5,mukin_bm,"age=2m, sex=F, microbiota=SPF, tissue=BM",SPF_2m_4,SPF_2m_5,death_rate,4,0.980854
6,mukin_sk,"age=19m, sex=F, microbiota=GF, tissue=SK",GF_19m_1,GF_19m_3,prolif_rate,4,0.715142
7,mukin_sk,"age=19m, sex=F, microbiota=GF, tissue=SK",GF_19m_1,GF_19m_3,death_rate,4,0.980076
8,mukin_sk,"age=19m, sex=F, microbiota=SPF, tissue=SK",SPF_19m_2,SPF_19m_3,prolif_rate,7,0.998373
9,mukin_sk,"age=19m, sex=F, microbiota=SPF, tissue=SK",SPF_19m_2,SPF_19m_3,death_rate,7,0.887006


In [ ]:
# rate reproducible checking
from pathlib import Path
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, wilcoxon
import math


REPRO_DIR = Path('/scratch/chenxi.sun/RA project/rates reproduce/rep_align_scatterplot copy')
PAIRWISE_RATE_FILE = REPRO_DIR / 'pairwise_rate_comparisons.csv'
PEARSON_FILE = REPRO_DIR / 'pairwise_rate_pearson_correlations.csv'
CONNECTED_DOT_PDF = REPRO_DIR / 'rate_reproducibility_connected_dot_plot.pdf'
PERMUTATION_GRID_PDF = REPRO_DIR / 'rate_reproducibility_permutation_null_grid.pdf'

N_PERMUTATIONS = 5000
MIN_PERMUTATION_CELLTYPES = 5
RANDOM_SEED = 20260628
PAIR_COLS = ['dataset', 'condition', 'mouse_a', 'mouse_b']
RATE_LABELS = {
    'prolif_rate': 'Proliferation',
    'death_rate': 'Death',
}
DATASET_COLORS = {
    'mukin_bm': '#4C78A8',
    'mukin_sk': '#F58518',
    'tms_bm': '#54A24B',
}

pairwise_rate_comparisons = pd.read_csv(PAIRWISE_RATE_FILE)
pairwise_rate_pearson_correlations = pd.read_csv(PEARSON_FILE)

# Task 1: paired Wilcoxon on pair-level Pearson r values.
pearson_wide = (
    pairwise_rate_pearson_correlations
    .pivot_table(
        index=PAIR_COLS + ['n_shared_celltypes'],
        columns='rate',
        values='pearson_r',
        aggfunc='first',
    )
    .rename(columns={'prolif_rate': 'prolif_r', 'death_rate': 'death_r'})
    .reset_index()
)
pearson_wide['delta'] = pearson_wide['prolif_r'] - pearson_wide['death_r']
wilcoxon_result = wilcoxon(pearson_wide['delta'], alternative='greater')
median_delta = pearson_wide['delta'].median()
n_prolif_gt_death = int((pearson_wide['delta'] > 0).sum())
n_pairs = pearson_wide.shape[0]

rng = np.random.default_rng(RANDOM_SEED)
jitter = rng.uniform(-0.045, 0.045, size=n_pairs)

fig, ax = plt.subplots(figsize=(4.8, 4.2), constrained_layout=True)
for i, row in pearson_wide.reset_index(drop=True).iterrows():
    color = DATASET_COLORS.get(row['dataset'], '#666666')
    x_values = np.array([0, 1], dtype=float) + jitter[i]
    y_values = [row['prolif_r'], row['death_r']]
    ax.plot(x_values, y_values, color=color, alpha=0.55, linewidth=1.2)
    ax.scatter(x_values, y_values, color=color, edgecolor='black', linewidth=0.35, s=38, zorder=3)

legend_handles = [
    Line2D([0], [0], color=color, marker='o', markeredgecolor='black', markeredgewidth=0.35, linewidth=1.4, label=dataset)
    for dataset, color in DATASET_COLORS.items()
    if dataset in set(pearson_wide['dataset'])
]
ax.legend(handles=legend_handles, frameon=False, loc='lower left', fontsize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Proliferation', 'Death'])
ax.set_ylabel('Pearson r')
ax.set_xlim(-0.22, 1.22)
y_min = min(float(pearson_wide[['prolif_r', 'death_r']].min().min()) - 0.04, 0.0)
y_max = min(float(pearson_wide[['prolif_r', 'death_r']].max().max()) + 0.04, 1.05)
ax.set_ylim(y_min, y_max)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#dddddd', linewidth=0.7, alpha=0.7)
ax.text(
    0.98,
    0.04,
    f'Median Δr = {median_delta:.3f}\nWilcoxon p = {wilcoxon_result.pvalue:.3g}\n{n_prolif_gt_death}/{n_pairs} pairs prolif r > death r',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=8.5,
    bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white', 'edgecolor': '#bbbbbb', 'alpha': 0.92},
)
fig.savefig(CONNECTED_DOT_PDF, bbox_inches='tight')
plt.close(fig)

task1_summary = pd.DataFrame([{
    'wilcoxon_statistic': wilcoxon_result.statistic,
    'p_value': wilcoxon_result.pvalue,
    'median_delta': median_delta,
    'n_pairs': n_pairs,
    'n_prolif_r_gt_death_r': n_prolif_gt_death,
}])

# Task 2: permutation null test for pair x rate entries with enough shared celltypes.
def _pair_mask(dataframe, row):
    mask = dataframe['dataset'].eq(row['dataset'])
    mask &= dataframe['condition'].eq(row['condition'])
    mask &= dataframe['mouse_a'].astype(str).eq(str(row['mouse_a']))
    mask &= dataframe['mouse_b'].astype(str).eq(str(row['mouse_b']))
    return mask

def _safe_pearson(vec_a, vec_b):
    if len(vec_a) < 2 or np.isclose(np.std(vec_a), 0) or np.isclose(np.std(vec_b), 0):
        return np.nan
    return pearsonr(vec_a, vec_b).statistic

qualifying_rate_rows = (
    pairwise_rate_pearson_correlations[pairwise_rate_pearson_correlations['n_shared_celltypes'] >= MIN_PERMUTATION_CELLTYPES]
    .sort_values(PAIR_COLS + ['rate'])
    .reset_index(drop=True)
)

permutation_rows = []
permutation_plot_data = []
rng = np.random.default_rng(RANDOM_SEED)

for _, rate_row in qualifying_rate_rows.iterrows():
    rate = rate_row['rate']
    pair_df = pairwise_rate_comparisons.loc[_pair_mask(pairwise_rate_comparisons, rate_row)].copy()
    pair_df = pair_df.dropna(subset=[f'{rate}_a', f'{rate}_b'])
    vec_a = pair_df[f'{rate}_a'].to_numpy(dtype=float)
    vec_b = pair_df[f'{rate}_b'].to_numpy(dtype=float)

    observed_r = _safe_pearson(vec_a, vec_b)
    if pd.isna(observed_r):
        null_distribution = np.full(N_PERMUTATIONS, np.nan)
        p_empirical = np.nan
    else:
        null_distribution = np.array([
            _safe_pearson(vec_a, rng.permutation(vec_b))
            for _ in range(N_PERMUTATIONS)
        ])
        p_empirical = np.nanmean(null_distribution >= observed_r)

    result_row = {
        'dataset': rate_row['dataset'],
        'condition': rate_row['condition'],
        'mouse_a': rate_row['mouse_a'],
        'mouse_b': rate_row['mouse_b'],
        'rate': rate,
        'observed_r': observed_r,
        'n_celltypes': len(vec_a),
        'p_empirical': p_empirical,
    }
    permutation_rows.append(result_row)
    permutation_plot_data.append((result_row, null_distribution))

n_panels = len(permutation_plot_data)
if n_panels:
    ncols = min(3, n_panels)
    nrows = math.ceil(n_panels / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 2.85 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, (row, null_distribution) in zip(axes, permutation_plot_data):
        valid_null = null_distribution[~np.isnan(null_distribution)]
        ax.hist(valid_null, bins=30, color='#9ecae1', edgecolor='white', linewidth=0.5)
        ax.axvline(row['observed_r'], color='#d62728', linestyle='--', linewidth=1.4)
        ax.set_title(f"{row['dataset']} | {row['condition']} | {row['rate']}", fontsize=7.5)
        ax.set_xlabel('Null Pearson r', fontsize=8)
        ax.set_ylabel('Count', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.text(
            0.97,
            0.95,
            f"obs r = {row['observed_r']:.3f}\np = {row['p_empirical']:.3g}\nn = {row['n_celltypes']}",
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=7.5,
            bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'white', 'edgecolor': '#bbbbbb', 'alpha': 0.9},
        )

    for ax in axes[n_panels:]:
        ax.set_visible(False)

    fig.savefig(PERMUTATION_GRID_PDF, bbox_inches='tight')
    plt.close(fig)
else:
    PERMUTATION_GRID_PDF = None

task2_summary = pd.DataFrame(permutation_rows)

print('Task 1 summary')
display(task1_summary)
print('Task 2 summary')
display(task2_summary)
print(f'Figure 1 saved to: {CONNECTED_DOT_PDF}')
print(f'Figure 2 saved to: {PERMUTATION_GRID_PDF}')


In [ ]:
# mukin_sk SPF/GF keratinocyte rate scatter plot
from pathlib import Path
from matplotlib.lines import Line2D

if 'tidyi_result_df' not in globals():
    tidyi_result_df = pd.read_csv('/scratch/chenxi.sun/RA project/rates reproduce/tidyi_result_df.csv')

KERATINOCYTE_CELLTYPES = [
    'keratinocyte basal early response',
    'keratinocyte basal label retain',
    'keratinocyte proliferating',
]
MICROBIOTA_ORDER = ['SPF', 'GF']
AGE_ORDER = ['2m', '19m']
CELLTYPE_MARKER_MAP = {
    'keratinocyte basal early response': 'o',
    'keratinocyte basal label retain': 's',
    'keratinocyte proliferating': '^',
}
AGE_COLOR_MAP = {
    '2m': '#4C78A8',
    '19m': '#F58518',
}

SK_KERATINOCYTE_SCATTER_PDF = Path(
    '/scratch/chenxi.sun/RA project/rates reproduce/rep_align_scatterplot copy/'
    'mukin_sk_keratinocyte_spf_gf_death_vs_prolif_scatter.pdf'
)

plot_df = tidyi_result_df[
    tidyi_result_df['dataset'].eq('mukin_sk')
    & tidyi_result_df['celltype'].isin(KERATINOCYTE_CELLTYPES)
    & tidyi_result_df['microbiota'].isin(MICROBIOTA_ORDER)
].copy()
plot_df['age'] = pd.Categorical(plot_df['age'], categories=AGE_ORDER, ordered=True)
plot_df['microbiota'] = pd.Categorical(plot_df['microbiota'], categories=MICROBIOTA_ORDER, ordered=True)
plot_df['celltype'] = pd.Categorical(plot_df['celltype'], categories=KERATINOCYTE_CELLTYPES, ordered=True)
plot_df = plot_df.sort_values(['celltype', 'microbiota', 'age', 'animal'])

fig, axes = plt.subplots(
    len(KERATINOCYTE_CELLTYPES),
    len(MICROBIOTA_ORDER),
    figsize=(7.8, 8.2),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for row_idx, celltype in enumerate(KERATINOCYTE_CELLTYPES):
    for col_idx, microbiota in enumerate(MICROBIOTA_ORDER):
        ax = axes[row_idx, col_idx]
        panel_df = plot_df[
            plot_df['celltype'].eq(celltype)
            & plot_df['microbiota'].eq(microbiota)
        ]
        for _, row in panel_df.iterrows():
            ax.scatter(
                row['death_rate'],
                row['prolif_rate'],
                s=62,
                marker=CELLTYPE_MARKER_MAP[celltype],
                color=AGE_COLOR_MAP[str(row['age'])],
                edgecolor='black',
                linewidth=0.4,
                alpha=0.88,
            )

        if row_idx == 0:
            ax.set_title(f'{microbiota} mice', fontsize=11)
        if col_idx == 0:
            ax.set_ylabel(f'{celltype}\nProliferation rate', fontsize=9)
        if row_idx == len(KERATINOCYTE_CELLTYPES) - 1:
            ax.set_xlabel('Death rate')

        ax.grid(color='#dddddd', linewidth=0.7, alpha=0.7)
        ax.spines[['top', 'right']].set_visible(False)

x_max = float(plot_df['death_rate'].max()) * 1.12 if len(plot_df) else 1
x_min = max(0, float(plot_df['death_rate'].min()) * 0.85) if len(plot_df) else 0
y_max = float(plot_df['prolif_rate'].max()) * 1.12 if len(plot_df) else 1
y_min = max(0, float(plot_df['prolif_rate'].min()) * 0.85) if len(plot_df) else 0
for ax in axes.ravel():
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

age_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='none',
        markerfacecolor=color,
        markeredgecolor='black',
        markeredgewidth=0.4,
        markersize=7,
        label=age,
    )
    for age, color in AGE_COLOR_MAP.items()
]

fig.legend(handles=age_handles, title='Age', loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8, title_fontsize=9)
fig.suptitle('mukin_sk keratinocyte rates by microbiota and cell type', fontsize=12)
fig.savefig(SK_KERATINOCYTE_SCATTER_PDF, bbox_inches='tight')
plt.show()

print(f'Saved figure to: {SK_KERATINOCYTE_SCATTER_PDF}')
plot_df[['microbiota', 'age', 'animal', 'celltype', 'death_rate', 'prolif_rate', 'cell_count']]
